In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from tqdm import tqdm
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='liac-arff')

In [10]:
NUM_INPUT = 784
NUM_OUTPUT = 50
DT = 0.005
T_MAX = 0.350
NUM_IMAGES = 10000

def assign_labels():
    print("Loading weights and training subset via sklearn...")
    weights = np.load('vdsp_weights.npy')
    
    images = mnist.data[:NUM_IMAGES] / 255.0
    labels = mnist.target[:NUM_IMAGES].astype(int)
    
    spike_counts = np.zeros((NUM_OUTPUT, 10))
    v_pre = np.zeros(NUM_INPUT)
    v_post = np.zeros(NUM_OUTPUT)
    
    # Track adaptation and clamps just like in training
    n_adapt = np.zeros(NUM_OUTPUT)
    refractory_pre = np.zeros(NUM_INPUT)
    refractory_post = np.zeros(NUM_OUTPUT)
    wta_clamp = np.zeros(NUM_OUTPUT)
    
    for img_idx, img in tqdm(enumerate(images), total=len(images), desc="Labeling"):
        v_pre.fill(0.0)
        v_post.fill(0.0)
        refractory_pre.fill(0.0)
        refractory_post.fill(0.0)
        wta_clamp.fill(0.0)
        img_label = labels[img_idx]
        
        for t in np.arange(0, T_MAX, DT):

            refractory_pre -= DT
            refractory_post -= DT
            wta_clamp -= DT
            
            # Input Neurons
            active_pre = refractory_pre <= 0
            v_pre[active_pre] += (DT / 0.03) * (-v_pre[active_pre] + img[active_pre] + 0.5)
            
            spikes_pre = v_pre >= 1.0
            v_pre[spikes_pre] = -1.0
            refractory_pre[spikes_pre] = 0.005
            
            # Output Neurons
            active_post = (refractory_post <= 0) & (wta_clamp <= 0)
            I_post = spikes_pre.astype(float) @ weights
            
            v_post[active_post] += (DT / 0.03) * (-v_post[active_post] + I_post[active_post] - n_adapt[active_post])
            n_adapt -= (DT / 1.0) * n_adapt
            
            spikes_post = v_post >= 1.0
            
            # Enforce Winner-Take-All during labeling
            if spikes_post.any():
                winner_idx = np.argmax(v_post)
                
                # Record the spike for the true winner only!
                spike_counts[winner_idx, img_label] += 1
                
                v_post[winner_idx] = 0.0
                refractory_post[winner_idx] = 0.005
                n_adapt[winner_idx] += 0.01
                
                wta_clamp[:] = 0.010
                wta_clamp[winner_idx] = 0.0
                v_post[wta_clamp > 0] = 0.0
                
    assigned_labels = np.argmax(spike_counts, axis=1)
    np.save('assigned_labels.npy', assigned_labels)
    print(f"Assigned Labels: {assigned_labels}")

In [14]:
assign_labels()

Loading weights and training subset via sklearn...


Labeling: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [01:21<00:00, 122.43it/s]

Assigned Labels: [3 9 7 9 0 4 2 9 5 6 1 3 6 4 4 1 1 8 7 9 4 4 8 5 3 3 2 9 5 6 9 6 1 5 2 8 5
 9 8 8 8 4 6 8 4 2 5 7 3 7]
